In [1]:
import numpy as np
from skimage.filters import gaussian
from PIL import Image, ImageEnhance
from Bio import SeqIO
import tkinter.filedialog as fd
import os.path

Splitting a large FASTQ file by tile

In [16]:
tile_sequences = []
fastq_path = 'F:/GENEMIND data/Lane02/202412161659_Lite3_5P240813019UY192671BX_zlf_L02.fq'
out_path = 'F:/GENEMIND data/Lane02/'
i = 1
first = True

for record in SeqIO.parse(fastq_path, "fastq"):
    if record is not None:
#         tile_num, x_pos, y_pos = get_pos(record)
        des = record.description
        tile_num = des.split(' ')[0].split(':')[4]
        if first:
            current_tile = tile_num
            first = False
        if tile_num == current_tile:
            tile_sequences.append(record)
        else:
            SeqIO.write(tile_sequences, out_path+current_tile+'.fq', "fastq")
            current_tile = tile_num
            tile_sequences = []
            tile_sequences.append(record)
            
SeqIO.write(tile_sequences, out_path+current_tile+'.fq', "fastq")                   


In [18]:
# SeqIO.write(tile_sequences, out_path+current_tile+'.fq', "fastq")        

263871

Generating FASTQ images for one tile

In [23]:
def library_index(template, strings, min_matches):
    
    row_sums = [sum(a == b for a, b in zip(row, template)) for row in strings]
    # Create the index of elements from Seq where row_sums is above the threshold
    index = [i for i, row_sum in enumerate(row_sums) if row_sum > min_matches]
    return index




tile_path = 'F:/GENEMIND data/Lane02/R001C002.fq'
x_coordinates = []
y_coordinates = []
sequences = []



for record in SeqIO.parse(tile_path, "fastq"):
    if record is not None:
        des = record.description
        des_fields = des.split(' ')[0].split(':')
        x_pos = int(des_fields[5])
        y_pos = int(des_fields[6])
        seq = str(record.seq)
        x_coordinates.append(x_pos)
        y_coordinates.append(y_pos)
        sequences.append(seq)
        
        
        
# Cas9 library        
library_seq =  'GGTCTCGCACAGCAGAAATCTCTACTGAGGTATAAAGATGAGACGCTGGAGTAAAAACGTTGGTTGGCT' 

idx = library_index(library_seq, sequences, 40)
x_coordinates_lib = [x_coordinates[i] for i in idx]
y_coordinates_lib = [y_coordinates[i] for i in idx]
sequences_lib = [sequences[i] for i in idx]


In [24]:
def generate_img(x, y, x_min, y_min, x_max, y_max, r, blurred, sigma):
    # x_min = min(x)
    # x_max = max(x)
    # y_min = min(y)
    # y_max = max(y)
    x_range = x_max - x_min
    y_range = y_max - y_min
    # x_range = 27994
    # y_range = 27174
    img = np.zeros(shape=(x_range, y_range))
    for i in range(0, len(x)):
        img[min(x_range-1,max(0,int(x[i]-r))):min(x_range-1,max(0,int(x[i]+r+1))),min(y_range-1,max(0,int(y[i]-r))):min(y_range-1,max(0,int(y[i]+r+1)))] = 200

    if blurred:
        img = gaussian(img, sigma=sigma)
    im = Image.fromarray(img)
    new_im = im.convert("L")
#     new_im.save(op_path)
    return new_im

fastq_image = generate_img(y_coordinates,x_coordinates, 0, 0, max(y_coordinates), max(x_coordinates), 1, True, 1)
fastq_image.save(tile_path[:-3]+'_FASTQ_all_image.png')

fastq_image_lib = generate_img(y_coordinates_lib,x_coordinates_lib, 0, 0, max(y_coordinates_lib), max(x_coordinates_lib), 1, True, 1)
fastq_image_lib.save(tile_path[:-3]+'_FASTQ_lib_image.png')



In [26]:
max(x_coordinates)

4107